import Pkg
Pkg.add("JuMP")
Pkg.add("HiGHS")

##### We use the storage variable to keep track of the number of planes in a city at time t
@constraint(model, [t in 1:time_periods, j in 1:cities],
    storage[t, j] == (t > 1 ? storage[t-1, j] : 0 ) + sum(y[t,i,j] for i in 1:cities) - sum(y[t,j,i] for i in 1:cities)
)

# Hot air - Airline

### 1 - Maximize the profit given hangers

In [48]:
# agiso@dtu.dk
using JuMP, HiGHS

cities = 4
time_periods = 3

D = zeros(Int, 3, 4, 4)

# period 1
D[1,:,:] = [
    0   50  53  14;
    84  0   80  21;
    17  58  0   40;
    31  79  34  0
]

# period 2
D[2,:,:] = [
    0   15  53  52;
    17  0   134 29;
    24  128 0   99;
    23  15  30  0
]

# period 3
D[3,:,:] = [
    0   3   16  9;
    48  0   104 48;
    62  92  0   68;
    13  15  21  0
]

P = [
    0    99   89   139;
    109  0    99   169;
    109  104  0    129;
    159  149  119  0
]

Ctakeoff = [
    0     5100  4400  8000;
    5100  0     11200 6900;
    4400  11200 0     5700;
    8000  6900  5700  0
]

hangers = [2, 1, 1, 0]
planes = 4

# Maximum number of passengers
M = 120

##### ----- Model ----- #####
model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

##### ----- Variables ----- #####
@variable(model, x[1:time_periods, 1:cities, 1:cities] >= 0)
@variable(model, y[1:time_periods, 1:cities, 1:cities] >= 0, Int)
@variable(model, z[1:time_periods+1, 1:cities]  >= 0, Int)

##### ----- objectives ----- #####
@objective(model, Max,
    sum(x[t, i, j] * P[i,j] for t in 1:time_periods, i in 1:1:cities, j in 1:1:cities) -
    sum(y[t, i, j] * Ctakeoff[i,j] for t in 1:time_periods, i in 1:1:cities, j in 1:1:cities)
)

##### ----- Constraints ----- #####
# The number of passengers should be lower than the demand
@constraint(model, [t in 1:time_periods, i in 1:1:cities, j in 1:1:cities],
    x[t,i,j] <= D[t,i,j]
)

# -----> BIG-M <-----
# Big-M link make y indicate wheter i plane is flown
@constraint(model, [t in 1:time_periods, i in 1:1:cities, j in 1:1:cities],
    x[t,i,j] <= M * y[t,i,j]
)


##### ----- Constraints - Keep track of aircraft ----- #####
# We can't fly from 1 city to itself - (No self loop)
@constraint(model, [i in 1:1:cities, t in 1:time_periods], x[t,i,i] == 0)

# Initial planes at start of period 1 must match overnight hangers
@constraint(model, [i in 1:cities], z[1,i] == hangers[i])
@constraint(model, [i in 1:cities], z[4,i] == hangers[i])

# Plane inventory balance:
# start of (t+1) = start of t + arrivals during t - departures during t
@constraint(model, [t in 1:time_periods, i in 1:cities],
    z[t+1,i] == z[t,i] + sum(y[t,j,i] for j in 1:cities) - sum(y[t,i,j] for j in 1:cities)
)

# Can't fly more planes out of a city than are available there at start of period t
@constraint(model, [t in 1:time_periods, i in 1:cities],
    sum(y[t,i,j] for j in 1:cities) <= z[t,i]
)

#### ----- Optimize ----- #####
optimize!(model)

println("Optimal solution:")
println("z = ", (objective_value(model)))

println("\n Passengers")
for t in 1:3
    println("\nTime", t)
    for i in 1:4
        println(value.(x[t,i,:]))
    end
end

println("\n Planes in air")
for t in 1:3
    println("\nTime", t)
    for i in 1:4
        println(value.(y[t,i,:]))
    end
end

println("\n Planes position")
for t in 1:3
    println("\nTime", t)
    println(value.(z[t,:]))
end



Optimal solution:
z = 12410.999999999995

 Passengers

Time1
[0.0, 50.00000000000002, 53.0, -0.0]
[84.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0]

Time2
[0.0, 0.0, 0.0, -0.0]
[-0.0, 0.0, 119.99999999999999, -0.0]
[0.0, 119.99999999999991, 0.0, 99.0]
[0.0, 0.0, -0.0, 0.0]

Time3
[0.0, 0.0, 0.0, 0.0]
[-0.0, 0.0, 0.0, 0.0]
[62.0, 0.0, 0.0, -0.0]
[-0.0, 0.0, 21.0, 0.0]

 Planes in air

Time1
[0.0, 1.0000000000000004, 1.0, 0.0]
[1.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0]

Time2
[0.0, 1.1102230246251565e-16, -0.0, 0.0]
[0.0, 0.0, 0.9999999999999999, 0.0]
[5.551115123125783e-16, 0.9999999999999993, 0.0, 1.0]
[0.0, -0.0, -0.0, 0.0]

Time3
[0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0]
[1.0, 0.0, 0.0, 0.0]
[0.0, -0.0, 1.0, 0.0]

 Planes position

Time1
[2.0, 1.0, 1.0, 0.0]

Time2
[0.9999999999999996, 1.0000000000000004, 2.0, 0.0]

Time3
[1.0, 1.0, 1.0, 1.0]


### 2 - Maximize the profit, but let the model choose the hangers

In [58]:
# agiso@dtu.dk
using JuMP, HiGHS

cities = 4
time_periods = 3

D = zeros(Int, 3, 4, 4)

# period 1
D[1,:,:] = [
    0   50  53  14;
    84  0   80  21;
    17  58  0   40;
    31  79  34  0
]

# period 2
D[2,:,:] = [
    0   15  53  52;
    17  0   134 29;
    24  128 0   99;
    23  15  30  0
]

# period 3
D[3,:,:] = [
    0   3   16  9;
    48  0   104 48;
    62  92  0   68;
    13  15  21  0
]

P = [
    0    99   89   139;
    109  0    99   169;
    109  104  0    129;
    159  149  119  0
]

Ctakeoff = [
    0     5100  4400  8000;
    5100  0     11200 6900;
    4400  11200 0     5700;
    8000  6900  5700  0
]

planes = 4

# Maximum number of passengers
M = 120

##### ----- Model ----- #####
model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

##### ----- Variables ----- #####
@variable(model, x[1:time_periods, 1:cities, 1:cities] >= 0)
@variable(model, y[1:time_periods, 1:cities, 1:cities] >= 0, Int)
@variable(model, z[1:time_periods+1, 1:cities]  >= 0, Int)
@variable(model, hangers[1:cities]  >= 0, Int)

##### ----- objectives ----- #####
@objective(model, Max,
    sum(x[t, i, j] * P[i,j] for t in 1:time_periods, i in 1:1:cities, j in 1:1:cities) -
    sum(y[t, i, j] * Ctakeoff[i,j] for t in 1:time_periods, i in 1:1:cities, j in 1:1:cities)
)

##### ----- Constraints ----- #####
# The number of passengers should be lower than the demand
@constraint(model, [t in 1:time_periods, i in 1:1:cities, j in 1:1:cities],
    x[t,i,j] <= D[t,i,j]
)

# -----> BIG-M <-----
# Big-M link make y indicate wheter i plane is flown
@constraint(model, [t in 1:time_periods, i in 1:1:cities, j in 1:1:cities],
    x[t,i,j] <= M * y[t,i,j]
)


##### ----- Constraints - Keep track of aircraft ----- #####
# We can't fly from 1 city to itself - (No self loop)
@constraint(model, [i in 1:1:cities, t in 1:time_periods], x[t,i,i] == 0)

# Initial planes at start of period 1 must match overnight hangers
@constraint(model, [i in 1:cities], z[1,i] - hangers[i] == 0)
@constraint(model, [i in 1:cities], z[4,i] - hangers[i] == 0)

# Plane inventory balance:
# start of (t+1) = start of t + arrivals during t - departures during t
@constraint(model, [t in 1:time_periods, i in 1:cities],
    z[t+1,i] == z[t,i] + sum(y[t,j,i] for j in 1:cities) - sum(y[t,i,j] for j in 1:cities)
)

# Can't fly more planes out of a city than are available there at start of period t
@constraint(model, [t in 1:time_periods, i in 1:cities],
    sum(y[t,i,j] for j in 1:cities) <= z[t,i]
)

#### ----- Optimize ----- #####
optimize!(model)

println("Optimal solution:")
println("z = ", (objective_value(model)))

println("\n Passengers")
for t in 1:3
    println("\nTime", t)
    for i in 1:4
        println(value.(x[t,i,:]))
    end
end

println("\n Planes in air")
for t in 1:3
    println("\nTime", t)
    for i in 1:4
        print(value.(y[t,i,:]))
    end
end

println("\n Planes position")
for t in 1:3
    println("\nTime", t)
    println(value.(z[t,:]))
end



Optimal solution:
z = 22368.0

 Passengers

Time1
[0.0, 0.0, 53.0, 0.0]
[84.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0]
[0.0, 79.0, 34.0, 0.0]

Time2
[0.0, 0.0, 53.0, 0.0]
[0.0, 0.0, 120.0, 0.0]
[0.0, 120.0, 0.0, 99.0]
[0.0, 0.0, 0.0, 0.0]

Time3
[0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0]
[62.0, 0.0, 0.0, 68.0]
[0.0, 0.0, 0.0, 0.0]

 Planes in air

Time1
[0.0, 0.0, 1.0, 0.0][1.0, 0.0, 0.0, 0.0][0.0, 0.0, 0.0, 0.0][0.0, 1.0, 1.0, 0.0]
Time2
[0.0, 0.0, 1.0, 0.0][0.0, 0.0, 1.0, 0.0][0.0, 1.0, 0.0, 1.0][0.0, 0.0, 0.0, 0.0]
Time3
[0.0, 0.0, 0.0, 0.0][0.0, 0.0, 0.0, 0.0][1.0, 0.0, 0.0, 1.0][0.0, 0.0, 0.0, 0.0]
 Planes position

Time1
[1.0, 1.0, 0.0, 2.0]

Time2
[1.0, 1.0, 2.0, 0.0]

Time3
[0.0, 1.0, 2.0, 1.0]
HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();


### 4 - Maximize the profit, but without planes that loose money

In [59]:
# agiso@dtu.dk
using JuMP, HiGHS

cities = 4
time_periods = 3

D = zeros(Int, 3, 4, 4)

# period 1
D[1,:,:] = [
    0   50  53  14;
    84  0   80  21;
    17  58  0   40;
    31  79  34  0
]

# period 2
D[2,:,:] = [
    0   15  53  52;
    17  0   134 29;
    24  128 0   99;
    23  15  30  0
]

# period 3
D[3,:,:] = [
    0   3   16  9;
    48  0   104 48;
    62  92  0   68;
    13  15  21  0
]

P = [
    0    99   89   139;
    109  0    99   169;
    109  104  0    129;
    159  149  119  0
]

Ctakeoff = [
    0     5100  4400  8000;
    5100  0     11200 6900;
    4400  11200 0     5700;
    8000  6900  5700  0
]

planes = 4

# Maximum number of passengers
M = 120

##### ----- Model ----- #####
model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

##### ----- Variables ----- #####
@variable(model, x[1:time_periods, 1:cities, 1:cities] >= 0)
@variable(model, y[1:time_periods, 1:cities, 1:cities] >= 0, Int)
@variable(model, z[1:time_periods+1, 1:cities]  >= 0, Int)
@variable(model, hangers[1:cities]  >= 0, Int)

##### ----- objectives ----- #####
@objective(model, Max,
    sum(x[t, i, j] * P[i,j] for t in 1:time_periods, i in 1:1:cities, j in 1:1:cities) -
    sum(y[t, i, j] * Ctakeoff[i,j] for t in 1:time_periods, i in 1:1:cities, j in 1:1:cities)
)

##### ----- Constraints ----- #####
# The number of passengers should be lower than the demand
@constraint(model, [t in 1:time_periods, i in 1:1:cities, j in 1:1:cities],
    x[t,i,j] <= D[t,i,j]
)

# -----> BIG-M <-----
# Big-M link make y indicate wheter i plane is flown
@constraint(model, [t in 1:time_periods, i in 1:1:cities, j in 1:1:cities],
    x[t,i,j] <= M * y[t,i,j]
)


##### ----- Constraints - Keep track of aircraft ----- #####
# We can't fly from 1 city to itself - (No self loop)
@constraint(model, [i in 1:1:cities, t in 1:time_periods], x[t,i,i] == 0)

# Initial planes at start of period 1 must match overnight hangers
@constraint(model, [i in 1:cities], z[1,i] - hangers[i] == 0)
@constraint(model, [i in 1:cities], z[4,i] - hangers[i] == 0)

# Plane inventory balance:
# start of (t+1) = start of t + arrivals during t - departures during t
@constraint(model, [t in 1:time_periods, i in 1:cities],
    z[t+1,i] == z[t,i] + sum(y[t,j,i] for j in 1:cities) - sum(y[t,i,j] for j in 1:cities)
)

# Can't fly more planes out of a city than are available there at start of period t
@constraint(model, [t in 1:time_periods, i in 1:cities],
    sum(y[t,i,j] for j in 1:cities) <= z[t,i]
)

##### ----- Constraints - No negative flights ----- #####
@constraint(model, [t in 1:time_periods, i in 1:1:cities, j in 1:1:cities],
    x[t,i,j] * P[i,j] - y[t,i,j] * Ctakeoff[i,j] >= 0 
)


#### ----- Optimize ----- #####
optimize!(model)

println("Optimal solution:")
println("z = ", (objective_value(model)))

println("\n Passengers")
for t in 1:3
    println("\nTime", t)
    for i in 1:4
        print("\n")
        for j in 1:4
            print(value.(x[t,i,j]) * P[i,j] - value.(y[t,i,j]) * Ctakeoff[i,j] )
            print(" ")
        end
    end
end


Optimal solution:
z = 20950.0

 Passengers

Time1

0.0 0.0 317.0 0.0 
4056.0 0.0 0.0 0.0 
0.0 0.0 0.0 0.0 
0.0 4871.0 0.0 0.0 
Time2

0.0 0.0 317.0 0.0 
0.0 0.0 680.0 0.0 
0.0 1280.0 0.0 7071.0 
0.0 0.0 0.0 0.0 
Time3

0.0 0.0 0.0 0.0 
0.0 0.0 0.0 0.0 
2358.0 0.0 0.0 0.0 
0.0 0.0 0.0 0.0 

# Wind turbines

A turbine can have any number of incoming connections,
but is limited to at most one outgoing connection

The number of incoming connections
(i.e., connected turbines) is limited by the capacity of the outgoing cable

In [97]:
using JuMP, HiGHS
include("Data/wind_farm_data.jl")


cost = [135, 250, 480]
capacity = [2, 6, 9]

cables = 3
turbine = 35
flow = 35 - 1
substaion = 35
substaion_cap = 4

model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

##### ----- Variables ----- #####
# i,j => ingoing, outgoing
@variable(model, x[1:staions, 1:staions], Bin) # Connections
@variable(model, y[1:cables, 1:staions, 1:staions], Bin) # Outgoing cable type
@variable(model, f[1:staions, 1:staions] >= 0, Int) # flow


##### ----- objectives ----- #####
@objective(model, Min,
    sum(y[c,i,j] * cost[c] * Distance[i,j] for c in 1:cables, i in 1:turbine, j in 1:turbine)
)

##### ----- Constraints : General ----- #####
# A turbine cannot connect to itself
@constraint(model, [i in 1:turbine],
    x[i,i] == 0
)  

# Every turbine has to have 1 outgoing edge
@constraint(model, [i in 1:turbine - 1],
    sum(x[i,j] for j in 1:turbine if j != i) == 1
)

# The substation shouldn't have an outgoing
@constraint(model, 
    sum(x[i,35] for i in 1:turbine) == 0
)

# Upto 4 incomming connection in the substation
@constraint(model, 
    sum(x[35,j] for j in 1:turbine) <= 4
)
@constraint(model, 
    sum(x[35,j] for j in 1:turbine) >= 1
)

# If a turbine has an outgoing edge, it needs a cable
@constraint(model, [i in 1:turbine, j in 1:turbine],
    sum(y[c,i,j] for c in 1:cables) >= x[i,j]
)  

##### ----- Constraints - Flow ----- #####
# Flow conservation
# Ingoing edges  + 1 = outgoing =>
# Ingoing - outgoing = 1
@constraint(model, [i in 1:turbine - 1],
    sum(f[i,j] for j in 1:turbine) - sum(f[j,i] for j in 1:turbine) == 1
)



# The sink needs to consume all flow
@constraint(model, 
    sum(f[35, j] for j in 1:stations) == flow
)

# Flow should only flow on connected edges (Big-M)
@constraint(model, [i in 1:turbine, j in 1:turbine],
    f[i,j] <=  x[i,j] * 100
)

#### ----- Optimize ----- #####
optimize!(model)

println("Optimal solution:")
println("z = ", (abs(objective_value(model))))

println("\nConnection:")
for i in 1:turbine
    println(value.(x[i,:]))
end

println("\nFlow:")
for i in 1:turbine
    println(value.(f[i,:]))
end

Optimal solution:
z = 0.0

Connection:
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 

In [ ]:
using JuMP, HiGHS, 
include("Data/wind_farm_data.jl")


cost = [135, 250, 480]
capacity = [2, 6, 9]

cables = 3
turbine = 35
flow = 35 - 1
substaion = 35
substaion_cap = 4

model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

##### ----- Variables ----- #####
# i,j => ingoing, outgoing
@variable(model, x[1:staions, 1:staions], Bin) # Connections
@variable(model, y[1:cables, 1:staions, 1:staions], Bin) # Outgoing cable type
@variable(model, f[1:staions, 1:staions] >= 0, Int) # flow


##### ----- objectives ----- #####
@objective(model, Min,
    sum(y[c,i,j] * cost[c] * Distance[i,j] for c in 1:cables, i in 1:turbine, j in 1:turbine)
)

##### ----- Constraints : General ----- #####
# A turbine cannot connect to itself
@constraint(model, [i in 1:turbine], x[i,i] == 0)  
@constraint(model, [i in 1:turbine], f[i,i] == 0)  

# Every turbine has to have 1 outgoing edge
@constraint(model, [i in 1:turbine - 1],
    sum(x[i,j] for j in 1:turbine if j != i) == 1
)

# The substation shouldn't have an outgoing
@constraint(model, 
    sum(x[i,35] for i in 1:turbine) == 0
)

# Upto 4 incomming connection in the substation
@constraint(model, 
    sum(x[35,j] for j in 1:turbine) <= 4
)
@constraint(model, 
    sum(x[35,j] for j in 1:turbine) >= 1
)

# If a turbine has an outgoing edge, it needs a cable
@constraint(model, [i in 1:turbine, j in 1:turbine],
    sum(y[c,i,j] for c in 1:cables) == x[i,j]
)  

##### ----- Constraints - Flow ----- #####
# Flow conservation
# Ingoing edges  + 1 = outgoing =>
# outgoing - Ingoing = 1
@constraint(model, [i in 1:turbine - 1],
    sum(f[j,i] for j in 1:turbine)- sum(f[i,j] for j in 1:turbine) == 1
)



# The sink needs to consume all flow
@constraint(model, 
    sum(f[35, j] for j in 1:stations) == flow
)

# Flow should only flow on connected edges (Big-M)
@constraint(model, [i in 1:turbine, j in 1:turbine],
    f[i,j] <=  x[i,j] * 100
)

#### ----- Optimize ----- #####
optimize!(model)

println("Optimal solution:")
println("z = ", (abs(objective_value(model))))

println("\nConnection:")
for i in 1:turbine
    println(value.(x[i,:]))
end

println("\nFlow:")
for i in 1:turbine
    println(value.(f[i,:]))
end